In [1]:
!git clone https://github.com/GyanRout/Major-Projects.git

Cloning into 'Major-Projects'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 25 (delta 1), reused 16 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 6.42 KiB | 6.42 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [2]:
import kagglehub
import shutil
import os

# 1. Download the latest version of the dataset
path = kagglehub.dataset_download("abhikjha/movielens-100k")
print("Original cached path:", path)

# 2. Define your target destination path in the cloned repo
target_dir = '/content/Major-Projects/dp_recommender_project/data/'

# 3. Create the 'data' folder (and any parent folders) if it doesn't exist
os.makedirs(target_dir, exist_ok=True)

# 4. Copy the unzipped contents from the cache to your new data folder
shutil.copytree(path, target_dir, dirs_exist_ok=True)

print(f"Dataset successfully copied to: {target_dir}")

100%|██████████| 971k/971k [00:00<00:00, 83.2MB/s]

Extracting files...
Original cached path: /root/.cache/kagglehub/datasets/abhikjha/movielens-100k/versions/1
Dataset successfully copied to: /content/Major-Projects/dp_recommender_project/data/


In [31]:
import pandas as pd
ratings_df = pd.read_csv('/content/Major-Projects/dp_recommender_project/data/ml-latest-small/ratings.csv')
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [32]:
threshold = 20
ratings_df['userId'].value_counts().min() # Calculates the number of movies each user have rated and then finding the minimum among them.
# As the minimum is 20, it is equal to our threshold so we need not remove any user from our data set.

20

In [33]:
ratings_df.movieId.value_counts().sort_index()

,count
movieId,
1,215
2,110
3,52
4,7
5,49
...,...
193581,1
193583,1
193585,1


In [37]:
# Creating a mapping dictionary

unique_user = ratings_df['userId'].unique()
unique_movies = ratings_df['movieId'].unique()

user_to_index = {old_id: new_id for new_id, old_id in enumerate(unique_user)}
movie_to_index = {old_id: new_id for new_id, old_id in enumerate(unique_movies)}

ratings_df['user_idx'] = ratings_df['userId'].map(user_to_index)
ratings_df['movie_idx'] = ratings_df['movieId'].map(movie_to_index)

ratings_df.head()

,userId,movieId,rating,timestamp,user_idx,movie_idx
0,1,1,4.0,964982703,0,0
1,1,3,4.0,964981247,0,1
2,1,6,4.0,964982224,0,2
3,1,47,5.0,964983815,0,3
4,1,50,5.0,964982931,0,4


In [42]:
import json

# Define the file paths where you want to save them
user_json_path = '/content/Major-Projects/dp_recommender_project/data/user_to_index.json'
movie_json_path = '/content/Major-Projects/dp_recommender_project/data/movie_to_index.json'

# Changing the content to int64 to save as json file
user_to_index_clean = { int(key): int(value) for key, value in user_to_index.items()}
movie_to_index_clean = { int(key): int(value) for key, value in movie_to_index.items()}

# Save the user mapping dictionary
with open(user_json_path, 'w') as f:
    json.dump(user_to_index_clean, f, indent=4)

# Save the movie mapping dictionary
with open(movie_json_path, 'w') as f:
    json.dump(movie_to_index_clean, f, indent=4)

print("Dictionaries successfully saved as JSON files!")

Dictionaries successfully saved as JSON files!


In [46]:
from sklearn.model_selection import train_test_split
# Splitting into traing and testing sets

X = ratings_df[['user_idx','movie_idx']]
y = ratings_df['rating']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [47]:
# concatinating the files and saving traing and testing files as csv

train_data = pd.concat([X_train, y_train], axis=1)
test_data = pd.concat([X_test, y_test], axis=1)

train_path = '/content/Major-Projects/dp_recommender_project/data/train_data.csv'
test_path = '/content/Major-Projects/dp_recommender_project/data/test_data.csv'

train_data.to_csv(train_path, index=False)
test_data.to_csv(test_path, index=False)

print(f"Traing and Testing data has been saved in {train_path} and {test_path}")

Traing and Testing data has been saved in /content/Major-Projects/dp_recommender_project/data/train_data.csv and /content/Major-Projects/dp_recommender_project/data/test_data.csv
